# Catching credit-card fraud — an honest model for Kaggle Playground S6E11

**Task.** Predict the probability that a card transaction is fraud (`is_fraud`). Kaggle scores submissions with ROC AUC.

**Read this first.** Playground data is **synthetic**: Kaggle generated it from a real-looking source dataset
(*Credit Card Fraud Detection 2026*). Patterns found here describe the generated data, not a real bank.

**Rules this notebook follows**
* 20% of the training data is locked away as an internal hold-out and scored **once**, at the end.
* All choices (model, features, Optuna) use 5-fold stratified cross-validation on the other 80%.
* No target encoding on full data, no statistics from the test set, `id` is never a feature.
* Every number reported anywhere in the project is written to `results/metrics.json` by this notebook.

**Kaggle version.** The project's `src/` modules are written below with `%%writefile`. Attach the competition data (Playground S6E11) — paths under `/kaggle/input` are found automatically.

Source repo: https://github.com/HussainAhmed75/credit-card-fraud-s6e11

In [ ]:
import os
os.makedirs('/kaggle/working/src', exist_ok=True)
os.chdir('/kaggle/working')

In [ ]:
%%writefile /kaggle/working/src/features.py
"""Loading, cleaning checks and features for Kaggle Playground S6E11
(Predicting Credit Card Fraud).

There is no time column in this data, so there is no "future" to leak.
The leakage risks here are different and are handled explicitly:
  * the row `id` is never a feature,
  * nothing is computed from the target outside a training fold
    (no target encoding on the full data; categoricals stay categorical),
  * no statistic is computed on the test set (no train+test frequency encoding),
  * the internal hold-out set is split off once and never used for any choice.
"""
from __future__ import annotations

import glob
import os

import numpy as np
import pandas as pd

TARGET = "is_fraud"
ID = "id"

CATEGORICAL = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
BOOLEAN = ["is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch",
           "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
NUMERIC_RAW = ["amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km",
               "card_age_months", "customer_age", "account_balance_usd", "cvv_retry_count",
               "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score",
               "prior_disputes"]
# row-level engineered features: each uses only the row itself
ENGINEERED = ["red_flag_count", "amount_to_balance", "log_amount", "is_night"]
NUMERIC = NUMERIC_RAW + BOOLEAN + ENGINEERED
FEATURES = NUMERIC + CATEGORICAL


def find_file(name: str, roots=("data", "../data")) -> str:
    for r in roots:
        p = os.path.join(r, name)
        if os.path.exists(p):
            return p
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(name)


def load(train_path: str, test_path: str):
    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)
    return tr, te


def prepare(df: pd.DataFrame, categories: dict | None = None) -> tuple[pd.DataFrame, dict]:
    """Add row-level features and fix dtypes. Category levels are taken from
    `categories` when given (so train and test share the same codes)."""
    d = df.copy()
    for c in BOOLEAN:
        d[c] = d[c].astype(float)
    d["red_flag_count"] = d[BOOLEAN].sum(axis=1)
    d["amount_to_balance"] = d["amount_usd"] / (d["account_balance_usd"].abs() + 1.0)
    d["log_amount"] = np.log1p(d["amount_usd"].clip(lower=0))
    d["is_night"] = d["time_of_day_hour"].isin([0, 1, 2, 3, 4, 5]).astype(float)
    cats = categories or {c: sorted(d[c].dropna().astype(str).unique()) for c in CATEGORICAL}
    for c in CATEGORICAL:
        d[c] = pd.Categorical(d[c].astype(str), categories=cats[c])
    if TARGET in d:
        d[TARGET] = d[TARGET].astype(int)
    return d, cats


def data_checks(tr: pd.DataFrame, te: pd.DataFrame) -> dict:
    cols = [c for c in tr.columns if c not in (ID, TARGET)]
    overlap = tr[cols].merge(te[cols].drop_duplicates(), on=cols, how="inner")
    return {
        "train_rows": int(len(tr)), "test_rows": int(len(te)),
        "n_raw_features": int(len(cols)),
        "missing_values_train": int(tr.isna().sum().sum()),
        "missing_values_test": int(te.isna().sum().sum()),
        "duplicate_rows_train": int(tr[cols].duplicated().sum()),
        "train_rows_identical_to_a_test_row": int(len(overlap)),
        "fraud_rate_pct": float(tr[TARGET].mean() * 100),
        "n_fraud": int(tr[TARGET].sum()),
    }


In [ ]:
%%writefile /kaggle/working/src/modeling.py
"""Splits, baselines, models, metrics, tuning and calibration."""
from __future__ import annotations

import time

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,
                             roc_auc_score)
from sklearn.model_selection import StratifiedKFold, train_test_split

from features import CATEGORICAL, FEATURES, NUMERIC, TARGET

SEED = 42
N_THREADS = 2
HOLDOUT_FRAC = 0.20
N_FOLDS = 5


# ---------------------------------------------------------------- splits
def holdout_split(d: pd.DataFrame):
    """One stratified 80/20 split. The 20% is touched once, at the very end."""
    dev_idx, hold_idx = train_test_split(d.index, test_size=HOLDOUT_FRAC, stratify=d[TARGET],
                                         random_state=SEED)
    return d.loc[dev_idx], d.loc[hold_idx]


def folds(y, n=N_FOLDS, seed=SEED):
    return list(StratifiedKFold(n_splits=n, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))


# ---------------------------------------------------------------- metrics
def metrics(y, p, top_frac=0.01) -> dict:
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    k = max(1, int(round(len(p) * top_frac)))
    top = np.argsort(-p, kind="mergesort")[:k]
    caught = y[top].sum()
    pc = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        "roc_auc": float(roc_auc_score(y, p)),
        "pr_auc": float(average_precision_score(y, p)),
        "recall_at_top1pct": float(caught / y.sum() * 100),
        "precision_at_top1pct": float(caught / k * 100),
        "brier": float(brier_score_loss(y, pc)),
        "log_loss": float(log_loss(y, pc)),
        "n": int(len(y)),
    }


def accuracy_at_05(y, p) -> float:
    return float(((np.asarray(p) >= 0.5).astype(int) == np.asarray(y)).mean() * 100)


# ---------------------------------------------------------------- baselines
def baseline_majority(d):
    """Always 'not fraud' (a constant score)."""
    return np.zeros(len(d))


def baseline_one_feature(d, col="merchant_risk_score"):
    """ONE strong feature used directly as the score."""
    return d[col].to_numpy(dtype=float)


def baseline_rule(d):
    """Hand-made analyst rule: count of red flags, ties broken by merchant risk."""
    return d["red_flag_count"].to_numpy() * 1000 + d["merchant_risk_score"].to_numpy()


# ---------------------------------------------------------------- models
def X_tree(d):
    return d[FEATURES]


def X_codes(d):
    X = d[FEATURES].copy()
    for c in CATEGORICAL:
        X[c] = X[c].cat.codes.astype(float)
    return X


def make_logreg():
    from sklearn.compose import ColumnTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    pre = ColumnTransformer([("num", StandardScaler(), NUMERIC),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=2000, C=1.0))])


def lgbm_params(**over):
    p = dict(objective="binary", learning_rate=0.05, n_estimators=4000, num_leaves=31,
             min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
             reg_lambda=1.0, random_state=SEED, n_jobs=N_THREADS, deterministic=True,
             force_row_wise=True, verbose=-1)
    p.update(over)
    return p


def fit_lgbm(Xtr, ytr, Xva=None, yva=None, **over):
    import lightgbm as lgb
    m = lgb.LGBMClassifier(**lgbm_params(**over))
    if Xva is not None:
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
              callbacks=[lgb.early_stopping(200, verbose=False)])
    else:
        m.fit(Xtr, ytr)
    return m


def fit_xgb(Xtr, ytr, Xva, yva):
    import xgboost as xgb
    m = xgb.XGBClassifier(n_estimators=4000, learning_rate=0.05, max_depth=5, subsample=0.8,
                          colsample_bytree=0.7, min_child_weight=5, tree_method="hist",
                          enable_categorical=True, max_cat_to_onehot=1, eval_metric="auc",
                          early_stopping_rounds=200, random_state=SEED, n_jobs=N_THREADS)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m


def fit_rf(Xtr, ytr, n_sub=200_000):
    from sklearn.ensemble import RandomForestClassifier
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(Xtr), size=min(n_sub, len(Xtr)), replace=False)
    m = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, max_features=0.4,
                               n_jobs=N_THREADS, random_state=SEED)
    m.fit(Xtr.iloc[idx], np.asarray(ytr)[idx])
    return m


def cv_oof(d, kind, fold_list, **over):
    """Out-of-fold predictions on the development set. Every model and every
    preprocessing step is fit on the training part of each fold only."""
    y = d[TARGET].to_numpy()
    oof = np.zeros(len(d))
    iters = []
    for tr_i, va_i in fold_list:
        a, b = d.iloc[tr_i], d.iloc[va_i]
        if kind == "logreg":
            m = make_logreg().fit(a[FEATURES], a[TARGET])
            oof[va_i] = m.predict_proba(b[FEATURES])[:, 1]
        elif kind == "rf":
            m = fit_rf(X_codes(a), a[TARGET])
            oof[va_i] = m.predict_proba(X_codes(b))[:, 1]
        elif kind == "xgb":
            m = fit_xgb(X_tree(a), a[TARGET], X_tree(b), b[TARGET])
            oof[va_i] = m.predict_proba(X_tree(b))[:, 1]
            iters.append(int(m.best_iteration))
        elif kind == "lgbm":
            m = fit_lgbm(X_tree(a), a[TARGET], X_tree(b), b[TARGET], **over)
            oof[va_i] = m.predict_proba(X_tree(b))[:, 1]
            iters.append(int(m.best_iteration_))
        else:
            raise ValueError(kind)
    return oof, iters


def timed(fn, *a, **k):
    t = time.time()
    out = fn(*a, **k)
    return out, round(time.time() - t, 1)


# ---------------------------------------------------------------- tuning
def tune_lgbm(d, n_trials=20, n_folds=3):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    fl = folds(d[TARGET].to_numpy(), n=n_folds, seed=SEED + 1)

    def objective(trial):
        over = dict(
            learning_rate=0.1,
            num_leaves=trial.suggest_int("num_leaves", 8, 128, log=True),
            min_child_samples=trial.suggest_int("min_child_samples", 20, 1000, log=True),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 50, log=True),
            min_split_gain=trial.suggest_float("min_split_gain", 0.0, 0.5),
        )
        oof, iters = cv_oof(d, "lgbm", fl, **over)
        trial.set_user_attr("mean_iter", float(np.mean(iters)))
        return roc_auc_score(d[TARGET], oof)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    return study


# ---------------------------------------------------------------- calibration
def reliability(y, p, n_bins=10):
    """Bins by predicted-probability quantile; returns mean predicted vs observed rate."""
    q = pd.qcut(pd.Series(p), q=n_bins, duplicates="drop")
    g = pd.DataFrame({"y": np.asarray(y), "p": p, "bin": q.values}).groupby("bin", observed=True)
    return g.agg(mean_pred=("p", "mean"), frac_pos=("y", "mean"), n=("y", "size")).reset_index(drop=True)


In [ ]:
%%writefile /kaggle/working/src/style.py
"""One consistent visual theme for every static chart (matplotlib) and the
Plotly dashboard."""
from __future__ import annotations

import matplotlib as mpl
import matplotlib.pyplot as plt

# palette: deep navy ink, one sand/gold accent (Dubai), muted greys
INK = "#1B2A41"
ACCENT = "#C8963E"      # highlight / the model
ACCENT_2 = "#2E7D7A"    # second series (e.g. Ready vs Off-Plan)
MUTED = "#A7B0BE"       # baselines / context
GRID = "#E6E9EF"
BG = "#FFFFFF"
GOOD = "#2E7D32"
BAD = "#B23A48"

SERIES = {"Off-Plan": ACCENT, "Ready": ACCENT_2}


def apply():
    mpl.rcParams.update({
        "figure.figsize": (10, 5.6),
        "figure.dpi": 110,
        "savefig.dpi": 160,
        "savefig.bbox": "tight",
        "figure.facecolor": BG,
        "axes.facecolor": BG,
        "axes.edgecolor": GRID,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 26,
        "axes.labelsize": 11,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.color": INK,
        "ytick.color": INK,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.frameon": False,
        "font.family": "DejaVu Sans",
        "text.color": INK,
    })


def title(ax, headline: str, sub: str | None = None):
    """Headline = the takeaway; sub = what is plotted."""
    ax.set_title(headline)
    if sub:
        ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=10.5,
                color="#5B6676", va="bottom", ha="left")


def source(fig, text="Source: Dubai Land Department open data via dubairealestatedata.com (CC BY 4.0)"):
    fig.text(0.01, -0.02, text, fontsize=8, color="#7A8494", ha="left")


def save(fig, path):
    fig.savefig(path)
    plt.close(fig)


PLOTLY_LAYOUT = dict(
    font=dict(family="DejaVu Sans, Arial, sans-serif", color=INK, size=13),
    paper_bgcolor=BG, plot_bgcolor=BG,
    xaxis=dict(gridcolor=GRID, zeroline=False),
    yaxis=dict(gridcolor=GRID, zeroline=False),
    margin=dict(l=60, r=30, t=80, b=50),
    colorway=[ACCENT, ACCENT_2, INK, MUTED, BAD, GOOD],
)


In [ ]:
%%writefile /kaggle/working/src/make_dashboard.py
"""Builds dashboard/fraud_dashboard.html — one self-contained offline Plotly page.
Every number comes from results/metrics.json or results/holdout_predictions.csv.gz."""
from __future__ import annotations

import html
import json
import os
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import style  # noqa: E402


def _layout(fig, title, sub=None, height=460):
    t = f"<b>{title}</b>" + (f"<br><span style='font-size:12px;color:#5B6676'>{sub}</span>" if sub else "")
    fig.update_layout(**style.PLOTLY_LAYOUT, title=dict(text=t, x=0.01, xanchor="left"), height=height)
    return fig


def build(root: str) -> str:
    M = json.load(open(os.path.join(root, "results", "metrics.json"), encoding="utf-8"))
    hp = pd.read_csv(os.path.join(root, "results", "holdout_predictions.csv.gz"))
    H = M["headline"]
    figs = []

    # 1 model comparison
    names = [n for n in ["Always 'not fraud'", "One feature (merchant risk score)", "Red-flag rule", "Random Forest",
                         "Logistic Regression", "XGBoost", "LightGBM", "LightGBM (tuned)",
                         "Blend: tuned LightGBM + LogReg (rank avg)"] if n in M["cv"]]
    fig = go.Figure()
    fig.add_trace(go.Bar(name="ROC AUC (CV)", x=names, y=[M["cv"][n]["roc_auc"] for n in names], marker_color=style.ACCENT_2,
                         text=[f"{M['cv'][n]['roc_auc']:.3f}" for n in names], textposition="outside"))
    fig.add_trace(go.Bar(name="PR AUC (CV)", x=names, y=[M["cv"][n]["pr_auc"] for n in names], marker_color=style.ACCENT,
                         text=[f"{M['cv'][n]['pr_auc']:.3f}" for n in names], textposition="outside"))
    fig.update_layout(barmode="group", yaxis_range=[0, 1.1], uniformtext=dict(minsize=9, mode="show"))
    figs.append(_layout(fig, "Models vs baselines (5-fold CV)", "PR AUC is the harder, more honest score when fraud is rare", 500))

    # 2 gain curve (hold-out) for chosen model's LightGBM probabilities
    y = hp["is_fraud"].to_numpy(); p = hp["p"].to_numpy()
    o = np.argsort(-p, kind="mergesort"); c = np.cumsum(y[o]) / y.sum(); x = np.arange(1, len(y) + 1) / len(y)
    k = int(len(x) * 0.2)
    fig = go.Figure(go.Scatter(x=x[:k] * 100, y=c[:k] * 100, mode="lines", line=dict(color=style.ACCENT, width=3),
                               hovertemplate="review %{x:.2f}% → catch %{y:.1f}% of fraud<extra></extra>"))
    fig.update_layout(xaxis_title="% of transactions reviewed (riskiest first)", yaxis_title="% of all fraud caught", yaxis_range=[0, 100])
    figs.append(_layout(fig, "How much fraud do you catch for the effort?", "Hold-out set, tuned LightGBM — hover to read any point"))

    # 3 explorer by merchant category
    cats = sorted(hp["merchant_category"].unique())
    fig = go.Figure(); buttons = []
    hp["bin"] = pd.qcut(hp["p"].rank(method="first"), 10, labels=False)
    for i, cat in enumerate(["All"] + cats):
        s = hp if cat == "All" else hp[hp.merchant_category == cat]
        g = s.groupby("bin").agg(pred=("p", "mean"), obs=("is_fraud", "mean"), n=("p", "size"))
        vis = i == 0
        fig.add_trace(go.Bar(x=g.index + 1, y=g["obs"] * 100, name="Observed fraud %", marker_color=style.ACCENT_2, visible=vis,
                             customdata=g["n"], hovertemplate="decile %{x}<br>observed %{y:.2f}%<br>%{customdata} txns<extra></extra>"))
        fig.add_trace(go.Scatter(x=g.index + 1, y=g["pred"] * 100, name="Predicted fraud %", mode="lines+markers",
                                 line=dict(color=style.ACCENT, width=3), visible=vis))
        mask = [False] * (2 * (len(cats) + 1)); mask[2 * i] = mask[2 * i + 1] = True
        buttons.append(dict(label=cat, method="update", args=[{"visible": mask},
                       {"title.text": f"<b>Results explorer — {html.escape(cat)}</b><br><span style='font-size:12px;color:#5B6676'>{len(s):,} hold-out transactions · {s.is_fraud.sum():,} fraud</span>"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right", y=1.2, yanchor="top")],
                      xaxis_title="Risk decile (1 = safest, 10 = riskiest; deciles of the whole hold-out)", yaxis_title="Fraud rate (%)")
    figs.append(_layout(fig, "Results explorer — All", f"{len(hp):,} hold-out transactions · {int(hp.is_fraud.sum()):,} fraud", 500))

    # 4 flags
    fr = pd.DataFrame(M["flag_rates"]).T.sort_values("with_pct")
    fig = go.Figure([go.Bar(y=fr.index, x=fr["without_pct"], orientation="h", name="flag absent", marker_color=style.MUTED),
                     go.Bar(y=fr.index, x=fr["with_pct"], orientation="h", name="flag present", marker_color=style.ACCENT,
                            text=[f"{v:.1f}%" for v in fr["with_pct"]], textposition="outside", cliponaxis=False)])
    fig.update_layout(barmode="group", xaxis_title="Fraud rate (%)")
    figs.append(_layout(fig, "Red flags and fraud rate", "Training data", 480))

    k1 = [(f"{H['holdout_auc']:.4f}", "ROC AUC on the locked hold-out"),
          (f"{H['holdout_pr_auc']:.3f}", "PR AUC on the hold-out"),
          (f"{H['holdout_recall_top1pct']:.1f}%", "of fraud caught reviewing the riskiest 1%"),
          (f"{H['holdout_precision_top1pct']:.1f}%", "of those reviewed are really fraud"),
          (f"{H['majority_holdout_accuracy']:.2f}%", "accuracy of always saying 'not fraud'")]
    parts = [f.to_html(full_html=False, include_plotlyjs="inline" if i == 0 else False, config={"displaylogo": False})
             for i, f in enumerate(figs)]
    case_rows = "".join(f"<tr><td>{c['case']}</td><td>{c['predicted_prob_pct']}%</td><td>{c['actual']}</td><td>{c['amount_usd']}</td>"
                        f"<td>{html.escape(c['merchant_category'])}</td><td>{html.escape(c['auth_method'])}</td><td>{c['red_flags']}</td>"
                        f"<td>{html.escape(', '.join(c['top_shap']))}</td></tr>" for c in M["case_studies"])
    page = f"""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Credit-card fraud model — dashboard</title><style>body{{font-family:'DejaVu Sans',Arial,sans-serif;color:{style.INK};margin:0;background:#F7F8FA}}
.wrap{{max-width:1150px;margin:0 auto;padding:28px}} .sub{{color:#5B6676;margin-bottom:22px}}
.kpis{{display:grid;grid-template-columns:repeat(5,1fr);gap:12px;margin-bottom:18px}} .kpi{{background:#fff;border-radius:10px;padding:14px;border:1px solid {style.GRID}}}
.kpi b{{font-size:24px;color:{style.ACCENT}}} .kpi span{{display:block;font-size:13px;color:#5B6676}}
.card{{background:#fff;border-radius:10px;padding:8px;border:1px solid {style.GRID};margin-bottom:16px}}
table{{border-collapse:collapse;width:100%;font-size:13px}} td,th{{border-bottom:1px solid {style.GRID};padding:6px;text-align:left}}
.warn{{background:#FFF7E6;border:1px solid #F0D9A8;border-radius:10px;padding:10px 14px;margin-bottom:16px;font-size:14px}}
.foot{{font-size:12px;color:#7A8494}}</style></head><body><div class="wrap">
<h1>Catching credit-card fraud</h1><div class="sub">Kaggle Playground S6E11 · {M['data']['train_rows']:,} training transactions · fraud rate {M['facts']['fraud_rate_pct']}%</div>
<div class="warn"><b>Synthetic data:</b> Kaggle generated this dataset from another one. Patterns describe the generated data, not a real bank.</div>
<div class="kpis">{''.join(f'<div class="kpi"><b>{a}</b><span>{b}</span></div>' for a, b in k1)}</div>
{''.join(f'<div class="card">{p}</div>' for p in parts)}
<div class="card" style="padding:16px"><h3>Case studies (hold-out)</h3><table><tr><th>Case</th><th>Predicted</th><th>Actual</th><th>Amount $</th><th>Merchant</th><th>Auth</th><th>Red flags</th><th>Top SHAP</th></tr>{case_rows}</table></div>
<p class="foot">Data: Kaggle Playground Series S6E11 (CC BY 4.0). Code: MIT. Built by Hussain Ahmed.</p></div></body></html>"""
    out = os.path.join(root, "dashboard", "fraud_dashboard.html")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    open(out, "w", encoding="utf-8").write(page)
    return out


if __name__ == "__main__":
    build(sys.argv[1] if len(sys.argv) > 1 else os.path.dirname(os.path.dirname(os.path.abspath(__file__))))


In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
ON_KAGGLE = os.path.exists("/kaggle/input")
ROOT = "/kaggle/working" if ON_KAGGLE else (os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))
for d_ in ["results", "figures", "models"]:
    os.makedirs(os.path.join(ROOT, d_), exist_ok=True)
from features import *
import modeling as mdl
import style
style.apply()
import matplotlib.pyplot as plt
from IPython.display import Image
R = lambda *p: os.path.join(ROOT, *p)
TRAIN = find_file("train.csv", roots=(os.path.join(ROOT, "data"),))
TEST = find_file("test.csv", roots=(os.path.join(ROOT, "data"),))
METRICS = {}
SRC = "Source: Kaggle Playground S6E11 (synthetic data, CC BY 4.0)"
print(TRAIN, TEST)

## 1. Load and check the data

In [ ]:
tr_raw, te_raw = load(TRAIN, TEST)
checks = data_checks(tr_raw, te_raw)
METRICS["data"] = checks
display(tr_raw.head(3).T)
display(pd.DataFrame({"dtype": tr_raw.dtypes.astype(str), "n_unique": tr_raw.nunique()}))
checks

**Cleaning decisions.** No missing values, no duplicate rows, and no training row identical to a test row
(see the checks above), so **no rows were removed**. The only changes: booleans → 0/1, categoricals kept as
categories (with the same levels for train and test), and four row-level features
(`red_flag_count`, `amount_to_balance`, `log_amount`, `is_night`) that use only the row itself.

In [ ]:
d, cats = prepare(tr_raw)
te, _ = prepare(te_raw, categories=cats)
y_all = d[TARGET]
print(d.shape, te.shape)

## 2. EDA — facts worth telling

In [ ]:
def rate(col):
    g = d.groupby(col, observed=True)[TARGET].agg(["mean", "size"])
    g["mean"] *= 100
    return g

auth = rate("auth_method"); merch = rate("merchant_category"); flags = rate("red_flag_count")
flag_rates = {c: {"with_pct": float(d.loc[d[c] == 1, TARGET].mean() * 100),
                  "without_pct": float(d.loc[d[c] == 0, TARGET].mean() * 100),
                  "n_with": int((d[c] == 1).sum())} for c in BOOLEAN}
from sklearn.metrics import roc_auc_score
single_auc = {c: float(max(roc_auc_score(y_all, d[c]), 1 - roc_auc_score(y_all, d[c]))) for c in NUMERIC_RAW}
best_single = max(single_auc, key=single_auc.get)
majority_acc = float((1 - y_all.mean()) * 100)
fl3 = d["red_flag_count"] >= 3
facts = {
    "fraud_rate_pct": round(checks["fraud_rate_pct"], 2),
    "majority_class_accuracy_pct": round(majority_acc, 2),
    "no_auth_fraud_pct": round(auth.loc["No Authentication", "mean"], 2),
    "biometric_fraud_pct": round(auth.loc["Biometric", "mean"], 2),
    "no_auth_vs_biometric_x": round(auth.loc["No Authentication", "mean"] / auth.loc["Biometric", "mean"], 1),
    "ip_mismatch_fraud_pct": round(flag_rates["ip_country_mismatch"]["with_pct"], 2),
    "ip_match_fraud_pct": round(flag_rates["ip_country_mismatch"]["without_pct"], 2),
    "riskiest_category": merch["mean"].idxmax(), "riskiest_category_pct": round(merch["mean"].max(), 2),
    "safest_category": merch["mean"].idxmin(), "safest_category_pct": round(merch["mean"].min(), 2),
    "zero_flags_fraud_pct": round(flags.loc[0, "mean"], 2),
    "three_plus_flags_fraud_pct": round(float(d.loc[fl3, TARGET].mean() * 100), 2),
    "three_plus_flags_share_pct": round(float(fl3.mean() * 100), 2),
    "best_single_feature": best_single, "best_single_feature_auc": round(single_auc[best_single], 4),
    "time_of_day_auc": round(single_auc["time_of_day_hour"], 4),
    "ai_scam_flag_fraud_pct": round(flag_rates["is_ai_generated_scam_attempt"]["with_pct"], 2),
    "ai_scam_flag_n": flag_rates["is_ai_generated_scam_attempt"]["n_with"],
}
METRICS["facts"] = facts
METRICS["flag_rates"] = {k: {kk: round(vv, 3) if isinstance(vv, float) else vv for kk, vv in v.items()} for k, v in flag_rates.items()}
METRICS["single_feature_auc"] = {k: round(v, 4) for k, v in single_auc.items()}
facts

In [ ]:
# Figure 1 - authentication
fig, ax = plt.subplots()
a = auth.sort_values("mean")
cols = [style.BAD if i == len(a) - 1 else (style.ACCENT_2 if i == 0 else style.MUTED) for i in range(len(a))]
ax.barh(a.index, a["mean"], color=cols)
for i, v in enumerate(a["mean"]):
    ax.text(v + 0.1, i, f"{v:.2f}%", va="center")
ax.set_xlabel("Fraud rate (%)"); ax.set_xlim(0, a["mean"].max() * 1.2)
style.title(ax, f"No authentication = {facts['no_auth_vs_biometric_x']}x the fraud rate of biometric",
            "Fraud rate by authentication method (training data)")
style.source(fig, SRC); style.save(fig, R("figures", "01_auth_method.png")); Image(R("figures", "01_auth_method.png"), width=760)

In [ ]:
# Figure 2 - red flags
fig, ax = plt.subplots(figsize=(10, 6))
fr = pd.DataFrame(flag_rates).T.sort_values("with_pct")
yy = np.arange(len(fr))
ax.barh(yy + 0.2, fr["without_pct"], height=0.4, color=style.MUTED, label="flag absent")
ax.barh(yy - 0.2, fr["with_pct"], height=0.4, color=style.ACCENT, label="flag present")
for i, (w, wo) in enumerate(zip(fr["with_pct"], fr["without_pct"])):
    ax.text(w + 0.2, i - 0.2, f"{w:.1f}%", va="center", fontsize=9)
ax.set_yticks(yy); ax.set_yticklabels([c.replace("_", " ") for c in fr.index])
ax.set_xlabel("Fraud rate (%)"); ax.set_xlim(0, fr["with_pct"].max() * 1.2); ax.legend(loc="lower right")
style.title(ax, f"An IP-country mismatch lifts fraud from {facts['ip_match_fraud_pct']}% to {facts['ip_mismatch_fraud_pct']}%",
            "Fraud rate with and without each yes/no red flag")
style.source(fig, SRC); style.save(fig, R("figures", "02_red_flags.png")); Image(R("figures", "02_red_flags.png"), width=760)

In [ ]:
# Figure 3 - merchant category
fig, ax = plt.subplots(figsize=(10, 6))
m_ = merch.sort_values("mean")
ax.barh(m_.index, m_["mean"], color=[style.ACCENT if v > 3 else style.MUTED for v in m_["mean"]])
for i, v in enumerate(m_["mean"]):
    ax.text(v + 0.1, i, f"{v:.2f}%", va="center", fontsize=9)
ax.set_xlabel("Fraud rate (%)"); ax.set_xlim(0, m_["mean"].max() * 1.2)
style.title(ax, f"{facts['riskiest_category']} are the riskiest purchase ({facts['riskiest_category_pct']}%)",
            "Fraud rate by merchant category")
style.source(fig, SRC); style.save(fig, R("figures", "03_merchant_category.png")); Image(R("figures", "03_merchant_category.png"), width=760)

In [ ]:
# Figure 4 - number of red flags
fig, ax = plt.subplots()
f_ = flags[flags["size"] >= 100]
ax.bar(f_.index.astype(int).astype(str), f_["mean"], color=style.ACCENT_2)
for i, (v, n) in enumerate(zip(f_["mean"], f_["size"])):
    ax.text(i, v + 1.5, f"{v:.1f}%\n(n={n:,})", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Number of red flags on the transaction"); ax.set_ylabel("Fraud rate (%)")
ax.set_ylim(0, f_["mean"].max() * 1.25)
style.title(ax, "Each extra red flag multiplies the risk",
            "Fraud rate by count of yes/no red flags (groups with at least 100 transactions)")
style.source(fig, SRC); style.save(fig, R("figures", "04_flag_count.png")); Image(R("figures", "04_flag_count.png"), width=760)

## 3. Validation design

* **Target:** `is_fraud` (1.67% positives).
* **Why not a time split?** The data has no date or ordering column, so there is no "future" to protect.
  Stratified K-fold is the right tool here; a time split would be invented.
* **Internal hold-out:** 20% stratified, locked until the end. **Model selection:** 5-fold stratified CV on the other 80%.
* **Metrics:** ROC AUC (Kaggle's metric), PR AUC (better for rare classes), and a business view:
  *if analysts review the riskiest 1% of transactions, what share of all fraud do they catch, and how many of the
  reviewed cases are really fraud?*

In [ ]:
dev, hold = mdl.holdout_split(d)
fl = mdl.folds(dev[TARGET].to_numpy())
assert set(dev.index).isdisjoint(hold.index)
METRICS["splits"] = {"dev_rows": int(len(dev)), "holdout_rows": int(len(hold)), "n_folds": mdl.N_FOLDS,
                     "dev_fraud_pct": round(float(dev[TARGET].mean() * 100), 3),
                     "holdout_fraud_pct": round(float(hold[TARGET].mean() * 100), 3),
                     "n_features": len(FEATURES)}
assert ID not in FEATURES and TARGET not in FEATURES
METRICS["splits"]

## 4. Baselines and models (5-fold CV on the development set)

In [ ]:
yd = dev[TARGET].to_numpy()
oofs, timings, iters = {}, {}, {}
oofs["Always 'not fraud'"] = mdl.baseline_majority(dev)
oofs["One feature (merchant risk score)"] = mdl.baseline_one_feature(dev)
oofs["Red-flag rule"] = mdl.baseline_rule(dev)
for name, kind in [("Logistic Regression", "logreg"), ("Random Forest", "rf"), ("XGBoost", "xgb"), ("LightGBM", "lgbm")]:
    (o, it), timings[name] = mdl.timed(mdl.cv_oof, dev, kind, fl)
    oofs[name] = o; iters[name] = it
cv_table = pd.DataFrame({k: mdl.metrics(yd, v) for k, v in oofs.items()}).T
cv_table.loc["Always 'not fraud'", "accuracy_at_0.5"] = mdl.accuracy_at_05(yd, oofs["Always 'not fraud'"])
for k in ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM"]:
    cv_table.loc[k, "accuracy_at_0.5"] = mdl.accuracy_at_05(yd, oofs[k])
cv_table = cv_table.sort_values("roc_auc", ascending=False)
display(cv_table.round(4)); print(timings)

## 5. Tune the best family with Optuna (3-fold CV inside the development set)

In [ ]:
ml = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM"]
best_family = cv_table.loc[ml, "roc_auc"].idxmax()
print("best family on CV:", best_family)
N_TRIALS = 20
study = mdl.tune_lgbm(dev, n_trials=N_TRIALS)
best = dict(study.best_params)
print(best, round(study.best_value, 5))
(o, it), timings["LightGBM (tuned)"] = mdl.timed(mdl.cv_oof, dev, "lgbm", fl, learning_rate=0.05, **best)
oofs["LightGBM (tuned)"] = o; iters["LightGBM (tuned)"] = it
# simple blend candidate: average of ranks of the two best different families
from scipy.stats import rankdata
blend = (rankdata(oofs["LightGBM (tuned)"]) + rankdata(oofs["Logistic Regression"])) / (2 * len(yd))
oofs["Blend: tuned LightGBM + LogReg (rank avg)"] = blend
cv_table = pd.DataFrame({k: mdl.metrics(yd, v) for k, v in oofs.items()}).T.sort_values("roc_auc", ascending=False)
for k in ml + ["LightGBM (tuned)", "Always 'not fraud'"]:
    cv_table.loc[k, "accuracy_at_0.5"] = mdl.accuracy_at_05(yd, oofs[k])
display(cv_table.round(4))
METRICS["cv"] = {k: {kk: (round(float(vv), 5) if pd.notna(vv) else None) for kk, vv in v.items()} for k, v in cv_table.to_dict("index").items()}
METRICS["tuning"] = {"family": "LightGBM", "n_trials": N_TRIALS, "tuning_folds": 3, "tuning_learning_rate": 0.1,
                     "best_params": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in best.items()},
                     "cv_auc_untuned": round(float(METRICS["cv"]["LightGBM"]["roc_auc"]), 5),
                     "cv_auc_tuned": round(float(METRICS["cv"]["LightGBM (tuned)"]["roc_auc"]), 5),
                     "mean_best_iteration": round(float(np.mean(iters["LightGBM (tuned)"])), 1)}
candidates = ["LightGBM (tuned)", "Blend: tuned LightGBM + LogReg (rank avg)"]
chosen = max(candidates, key=lambda k: METRICS["cv"][k]["roc_auc"])
METRICS["chosen_model"] = chosen
print("chosen on CV:", chosen)

## 6. The locked hold-out (scored once)

In [ ]:
n_iter = int(round(np.mean(iters["LightGBM (tuned)"]) * 1.1))
final_lgbm = mdl.fit_lgbm(mdl.X_tree(dev), dev[TARGET], learning_rate=0.05, n_estimators=n_iter, **best)
p_lgbm = final_lgbm.predict_proba(mdl.X_tree(hold))[:, 1]
final_lr = mdl.make_logreg().fit(dev[FEATURES], dev[TARGET])
p_lr = final_lr.predict_proba(hold[FEATURES])[:, 1]
p_blend = (rankdata(p_lgbm) + rankdata(p_lr)) / (2 * len(p_lgbm))
yh = hold[TARGET].to_numpy()
hold_preds = {"Always 'not fraud'": mdl.baseline_majority(hold), "One feature (merchant risk score)": mdl.baseline_one_feature(hold),
              "Red-flag rule": mdl.baseline_rule(hold), "Logistic Regression": p_lr, "LightGBM (tuned)": p_lgbm,
              "Blend: tuned LightGBM + LogReg (rank avg)": p_blend}
hold_table = pd.DataFrame({k: mdl.metrics(yh, v) for k, v in hold_preds.items()}).T.sort_values("roc_auc", ascending=False)
for k in ["Always 'not fraud'", "Logistic Regression", "LightGBM (tuned)"]:
    hold_table.loc[k, "accuracy_at_0.5"] = mdl.accuracy_at_05(yh, hold_preds[k])
display(hold_table.round(4))
METRICS["holdout"] = {k: {kk: (round(float(vv), 5) if pd.notna(vv) else None) for kk, vv in v.items()} for k, v in hold_table.to_dict("index").items()}
ch = METRICS["holdout"][chosen]; rule = METRICS["holdout"]["Red-flag rule"]; one = METRICS["holdout"]["One feature (merchant risk score)"]
METRICS["headline"] = {"chosen_model": chosen, "holdout_auc": ch["roc_auc"], "holdout_pr_auc": ch["pr_auc"],
    "holdout_recall_top1pct": ch["recall_at_top1pct"], "holdout_precision_top1pct": ch["precision_at_top1pct"],
    "rule_holdout_auc": rule["roc_auc"], "rule_recall_top1pct": rule["recall_at_top1pct"],
    "one_feature_holdout_auc": one["roc_auc"], "cv_auc_chosen": METRICS["cv"][chosen]["roc_auc"],
    "lgbm_holdout_accuracy_at_05": METRICS["holdout"]["LightGBM (tuned)"]["accuracy_at_0.5"],
    "majority_holdout_accuracy": METRICS["holdout"]["Always 'not fraud'"]["accuracy_at_0.5"],
    "lift_top1pct_x": round(ch["precision_at_top1pct"] / (yh.mean() * 100), 1),
    "logreg_holdout_auc": METRICS["holdout"]["Logistic Regression"]["roc_auc"],
    "lgbm_holdout_auc": METRICS["holdout"]["LightGBM (tuned)"]["roc_auc"]}
METRICS["headline"]

In [ ]:
# gain curve: share of all fraud caught vs share of transactions reviewed
def gain(y, p):
    o = np.argsort(-p, kind="mergesort"); c = np.cumsum(y[o]) / y.sum()
    x = np.arange(1, len(y) + 1) / len(y)
    return x, c
review = [0.005, 0.01, 0.02, 0.05, 0.10]
gains = {}
for name in [chosen, "Red-flag rule", "One feature (merchant risk score)"]:
    x, c = gain(yh, hold_preds[name])
    gains[name] = {f"{int(r*1000)/10}%": round(float(c[int(len(c) * r) - 1] * 100), 2) for r in review}
METRICS["gain_holdout"] = gains
fig, ax = plt.subplots()
for name, col, lw in [(chosen, style.ACCENT, 3), ("Red-flag rule", style.MUTED, 2), ("One feature (merchant risk score)", style.INK, 1.5)]:
    x, c = gain(yh, hold_preds[name])
    ax.plot(x[: int(len(x) * 0.2)] * 100, c[: int(len(x) * 0.2)] * 100, color=col, lw=lw, label=name)
ax.axvline(1, color=style.GRID, ls="--")
ax.set_xlabel("% of transactions reviewed (riskiest first)"); ax.set_ylabel("% of all fraud caught"); ax.set_ylim(0, 100)
ax.legend(loc="lower right")
g1 = gains[chosen]["1.0%"]
style.title(ax, f"Reviewing the riskiest 1% catches {g1:.1f}% of all fraud",
            "Hold-out set (scored once). Higher and further left is better")
style.source(fig, SRC); style.save(fig, R("figures", "05_gain_curve.png")); Image(R("figures", "05_gain_curve.png"), width=760)

In [ ]:
# Figure 6 - model comparison (CV)
order = ["Always 'not fraud'", "One feature (merchant risk score)", "Red-flag rule", "Random Forest", "Logistic Regression",
         "XGBoost", "LightGBM", "LightGBM (tuned)", "Blend: tuned LightGBM + LogReg (rank avg)"]
vals = [METRICS["cv"][o]["roc_auc"] for o in order]
fig, ax = plt.subplots(figsize=(10, 6.2))
cols = [style.MUTED if i < 3 else style.ACCENT_2 for i in range(len(order))]
cols[order.index(chosen)] = style.ACCENT
ax.barh(order[::-1], vals[::-1], color=cols[::-1])
for i, v in enumerate(vals[::-1]):
    ax.text(v + 0.003, i, f"{v:.4f}", va="center", fontsize=9)
ax.set_xlim(0.45, 1.0); ax.set_xlabel("ROC AUC, 5-fold CV on the development set (higher is better)")
style.title(ax, "Trees and a plain logistic regression end up close",
            f"Grey = baselines. Gold = chosen model. Logistic regression {METRICS['cv']['Logistic Regression']['roc_auc']:.4f} vs tuned LightGBM {METRICS['cv']['LightGBM (tuned)']['roc_auc']:.4f}")
style.source(fig, SRC); style.save(fig, R("figures", "06_model_comparison.png")); Image(R("figures", "06_model_comparison.png"), width=760)

## 7. Calibration — can the probabilities be trusted?

In [ ]:
rel = mdl.reliability(yh, p_lgbm, n_bins=10)
METRICS["calibration_lgbm_holdout"] = {"brier": METRICS["holdout"]["LightGBM (tuned)"]["brier"],
    "brier_constant_rate": round(float(np.mean((yh - yh.mean()) ** 2)), 5),
    "top_bin_mean_pred_pct": round(float(rel["mean_pred"].iloc[-1] * 100), 2),
    "top_bin_observed_pct": round(float(rel["frac_pos"].iloc[-1] * 100), 2),
    "max_abs_gap_pct_points": round(float((rel["mean_pred"] - rel["frac_pos"]).abs().max() * 100), 2)}
fig, ax = plt.subplots(figsize=(7.5, 7))
lim = max(rel["mean_pred"].max(), rel["frac_pos"].max()) * 100 * 1.1
ax.plot([0, lim], [0, lim], ls="--", color=style.INK, lw=1)
ax.plot(rel["mean_pred"] * 100, rel["frac_pos"] * 100, marker="o", color=style.ACCENT, lw=2.5)
ax.set_xlabel("Mean predicted fraud probability (%)"); ax.set_ylabel("Observed fraud rate (%)")
cal = METRICS["calibration_lgbm_holdout"]
style.title(ax, f"Probabilities are honest: worst bin is off by {cal['max_abs_gap_pct_points']} points",
            "Hold-out, tuned LightGBM, 10 equal-size bins. Dashed = perfect calibration")
style.source(fig, SRC); style.save(fig, R("figures", "07_calibration.png")); Image(R("figures", "07_calibration.png"), width=600)

## 8. Explanation — SHAP and real cases

In [ ]:
import shap
expl = shap.TreeExplainer(final_lgbm)
sample = hold.sample(5000, random_state=mdl.SEED)
sv = expl.shap_values(mdl.X_tree(sample))
sv = sv[1] if isinstance(sv, list) else sv
imp = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values(ascending=False)
METRICS["shap_top10"] = {k: round(float(v), 4) for k, v in imp.head(10).items()}
fig, ax = plt.subplots(figsize=(10, 6))
t10 = imp.head(10).iloc[::-1]
ax.barh([k.replace("_", " ") for k in t10.index], t10.values, color=[style.ACCENT if i >= 7 else style.MUTED for i in range(10)])
ax.set_xlabel("Mean |SHAP| (impact on log-odds of fraud)")
style.title(ax, f"'{imp.index[0].replace('_', ' ')}' drives the model most",
            "Top 10 features by mean absolute SHAP, 5,000 hold-out transactions")
style.source(fig, SRC); style.save(fig, R("figures", "08_shap.png")); Image(R("figures", "08_shap.png"), width=760)

In [ ]:
hh = hold.assign(p=p_lgbm)
thr = np.quantile(p_lgbm, 0.99)          # the "review the riskiest 1%" threshold
caught = hh[(hh[TARGET] == 1) & (hh.p >= thr)].sort_values("p", ascending=False)
missed = hh[(hh[TARGET] == 1) & (hh.p < thr)].sort_values("p")
false_alarm = hh[(hh[TARGET] == 0) & (hh.p >= thr)].sort_values("p", ascending=False)
picks = [("caught_fraud", caught.iloc[len(caught) // 2]), ("missed_fraud", missed.iloc[len(missed) // 2]),
         ("false_alarm", false_alarm.iloc[len(false_alarm) // 2])]
cases = []
for label, r in picks:
    s = expl.shap_values(mdl.X_tree(hh.loc[[r.name]]))
    s = (s[1] if isinstance(s, list) else s)[0]
    top = pd.Series(s, index=FEATURES).sort_values(key=np.abs, ascending=False).head(3)
    cases.append({"case": label, "id": int(r[ID]), "predicted_prob_pct": round(float(r.p * 100), 2),
                  "actual": int(r[TARGET]), "amount_usd": float(r.amount_usd), "merchant_category": str(r.merchant_category),
                  "auth_method": str(r.auth_method), "red_flags": int(r.red_flag_count),
                  "merchant_risk_score": float(r.merchant_risk_score),
                  "top_shap": [f"{k} ({'+' if v > 0 else '-'})" for k, v in top.items()]})
METRICS["review_threshold_prob_pct"] = round(float(thr * 100), 3)
METRICS["case_studies"] = cases
pd.DataFrame(cases).T

## 9. Final model on all training data → Kaggle submission

In [ ]:
full_lgbm = mdl.fit_lgbm(mdl.X_tree(d), d[TARGET], learning_rate=0.05, n_estimators=n_iter, **best)
pt_lgbm = full_lgbm.predict_proba(mdl.X_tree(te))[:, 1]
if chosen.startswith("Blend"):
    full_lr = mdl.make_logreg().fit(d[FEATURES], d[TARGET])
    pt_lr = full_lr.predict_proba(te[FEATURES])[:, 1]
    pred = (rankdata(pt_lgbm) + rankdata(pt_lr)) / (2 * len(pt_lgbm))
else:
    pred = pt_lgbm
sub = pd.DataFrame({ID: te_raw[ID], TARGET: pred})
out_sub = "/kaggle/working/submission.csv" if ON_KAGGLE else R("results", "submission.csv")
sub.to_csv(out_sub, index=False)
METRICS["submission"] = {"rows": int(len(sub)), "n_estimators": n_iter, "model": chosen}
full_lgbm.booster_.save_model(R("models", "lgbm_full.txt"))
defaults = {c: float(d[c].median()) for c in NUMERIC_RAW}
defaults.update({c: str(d[c].mode()[0]) for c in CATEGORICAL})
ranges = {c: [float(d[c].quantile(0.001)), float(d[c].quantile(0.999))] for c in NUMERIC_RAW}
json.dump({"features": FEATURES, "categorical": CATEGORICAL, "categories": cats, "boolean": BOOLEAN,
           "numeric_raw": NUMERIC_RAW, "defaults": defaults, "ranges": ranges,
           "review_threshold_prob": float(thr), "fraud_rate": float(d[TARGET].mean())}, open(R("models", "model_meta.json"), "w"), indent=1)
hh[[ID, TARGET, "p", "amount_usd", "merchant_category", "auth_method", "channel", "red_flag_count",
    "merchant_risk_score", "velocity_score", "cvv_retry_count"]].round(5).to_csv(R("results", "holdout_predictions.csv.gz"), index=False)
sub.head()

In [ ]:
METRICS["config"] = {"holdout_frac_pct": 20, "n_folds": 5, "optuna_trials": N_TRIALS, "top_review_pct": 1,
                     "rf_subsample_rows": 200000, "shap_sample_rows": 5000, "seed": mdl.SEED, "calibration_bins": 10,
                     "review_grid_pct": [0.5, 1, 2, 5, 10]}
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, (float, np.floating)): return None if np.isnan(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
json.dump(_clean(METRICS), open(R("results", "metrics.json"), "w", encoding="utf-8"), indent=2, ensure_ascii=False, sort_keys=True)
import make_dashboard
make_dashboard.build(ROOT)
print(json.dumps(METRICS["headline"], indent=1))

## Limitations
* **Synthetic data.** Kaggle generated it from another dataset; real fraud is messier and changes over time.
* No time column, so the model cannot be tested on "next month"; in a real bank that test would be required.
* ROC AUC hides the rarity of fraud; PR AUC and the "review 1%" numbers above are the ones a fraud team would feel.
* The decision threshold (review the riskiest 1%) is an example, not a recommendation — it depends on analyst capacity and costs.